# Persona Engine ML Lab

**Environment:** Google Colab
**Purpose:** Setup ML runtime, verify GPU acceleration, install dependencies, and inspect environment.
**Input:** `ml/requirements-colab.txt`, `ml/configs/base.yaml`
**Output:** Verified ML environment, loaded configuration, diagnostic report

---

### Workflow Overview
```text
Local Machine (Code & Architecture)
        |
        v (Git Push)
GitHub Repository
        |
        v (Git Clone)
Google Colab GPU Runtime
        |
        +--> 00_colab_setup.ipynb        (Environment bootstrap)
        +--> 01_dataset_analysis.ipynb   (EDA on conversation data)
        +--> 02_preprocessing.ipynb      (Cleaning, validation, splitting)
        +--> 03_annotation.ipynb         (Behavioral tagging)
        +--> 04_baseline_inference.ipynb (Zero-shot baseline testing)
        +--> 05_training.ipynb           (QLoRA fine-tuning)
        +--> 06_evaluation.ipynb         (Persona alignment evaluation)
```

## 1. System Diagnostics & Hardware Verification

In [ ]:
import sys
import platform
import os
import torch

print("Python Version :", platform.python_version())
print("PyTorch Version:", torch.__version__)
print("CUDA Available :", torch.cuda.is_available())

gpu_name = "N/A (CPU)"
gpu_mem = "N/A"
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = f"{torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB"
    print(f"GPU Detected   : {gpu_name} ({gpu_mem})")
else:
    print("WARNING: Running on CPU. If running on Google Colab, change runtime to T4 or A100 GPU.")

## 2. Clone Repository or Set Workspace Path
If running on Google Colab, mount Google Drive and clone your repository.
For private repositories, use Colab Secrets (`from google.colab import userdata; token = userdata.get('GITHUB_TOKEN')`) instead of hardcoding tokens.

In [ ]:
import os
from pathlib import Path

# Detect if running inside Google Colab
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("Running inside Google Colab environment.")
    # Set workspace path
    workspace_dir = Path("/content/persona-engine")
    if not workspace_dir.exists():
        # Clone repo instructions:
        # !git clone https://github.com/<your-username>/persona-engine.git /content/persona-engine
        print("Please clone your repository into /content/persona-engine")
    ml_dir = workspace_dir / "ml"
else:
    # Local workspace resolution
    ml_dir = Path.cwd().resolve()
    if ml_dir.name != "ml" and (ml_dir / "ml").is_dir():
        ml_dir = ml_dir / "ml"

print("ML Directory:", ml_dir)
if str(ml_dir) not in sys.path:
    sys.path.insert(0, str(ml_dir))

## 3. Install Dependencies
Installs GPU-optimized dependencies on Colab.

In [ ]:
# In Colab, run:
# !pip install -r {ml_dir}/requirements-colab.txt

import yaml
import jsonschema
print("Core configuration and schema packages verified.")

## 4. Load Configuration & Verify Directories

In [ ]:
from src.config import load_config

# Load base config with Colab/local environment detection
env_target = "colab" if IS_COLAB else "local"
config = load_config(environment=env_target)
print("Environment :", config.environment)
print("Base Model  :", config.model.name)
print("Data Root   :", config.paths.data_root)

# Verify required folders exist
for folder_name, folder_path in config.paths.items():
    p = Path(folder_path)
    status = "OK" if p.exists() else "CREATED"
    p.mkdir(parents=True, exist_ok=True)
    print(f"  [{status}] {folder_name}: {p}")

## 5. Verify Sample Dataset

In [ ]:
from src.data.validator import validate_dataset_file

sample_path = Path(config.paths.sample_data) / "sample_conversations.jsonl"
print(f"Checking sample dataset at: {sample_path}")
if sample_path.exists():
    val_result = validate_dataset_file(sample_path)
    print(val_result.summary())
else:
    print("Sample dataset not found. Please verify paths.")

## 6. Environment Summary

In [ ]:
import transformers

print("========================================")
print("PERSONA ENGINE ML LAB")
print("========================================")
print("Python        :", platform.python_version())
print("PyTorch       :", torch.__version__)
print("Transformers  :", transformers.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU           :", gpu_name)
print("GPU Memory    :", gpu_mem)
print("ML Directory  :", ml_dir)
print("Environment   :", env_target.upper())
print("Status        : READY FOR EXPERIMENTATION")
print("========================================")